<a href="https://colab.research.google.com/github/kriss-spy/pr-course-prj/blob/main/EvTrack/code/SDSTrack/SDSTrack_VisEvent_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDSTrack on VisEvent — Colab Notebook

This notebook reproduces **SDSTrack** evaluation on the **VisEvent** dataset for the Pattern Recognition course project (Topic #65).

## Workflow
1. **Environment Setup** — Mount Drive, install deps, clone SDSTrack, apply patches
2. **Dataset Preparation** — Extract VisEvent train/test sets from Google Drive
3. **Model Preparation** — Set up pretrained checkpoint, apply PyTorch 2.x compatibility fixes
4. **Evaluation** — Run tracking on test set and compute Success/Precision metrics

## Expected Results
- **Success AUC:** ~0.62
- **Precision (20px):** ~0.74

## Tips
- All setup cells are **idempotent** — safe to re-run.
- Long-running cells (extraction) show progress and have keepalive.
- Use *Runtime > Factory reset runtime* if you need a completely fresh start.

## 本 Notebook 解决什么问题？

**课题：** 基于事件相机的目标跟踪（Topic #65）

### 核心难点
传统帧相机在高速运动、极端光照变化等场景下存在运动模糊、信息延迟等问题。事件相机以像素级异步触发机制提供微秒级时间分辨率与高动态范围，但如何将其稀疏事件流与传统图像有效融合，是提升跟踪鲁棒性的关键。

### SDSTrack 复现痛点
- **环境断层**：原代码基于 PyTorch 1.11 + Python 3.8，Colab 已升级到 PyTorch 2.x + Python 3.10+
- **路径硬编码**：数据集、模型路径写死为作者本地服务器路径
- **checkpoint 兼容性**：PyTorch 2.6+ 默认 `weights_only=True`，无法加载旧模型
- **数据规模**：VisEvent 数据集 232 GB，需从 Google Drive 分卷解压并管理空间

### 本 Notebook 的解决方案
| 问题 | 解决方案 |
|------|----------|
| 环境不兼容 | 自动检测并应用 PyTorch 2.x / Python 3.10+ 兼容性补丁（`collections.abc`、`torch._six`、`weights_only` 等） |
| 路径错误 | 自动重写 `local.py` 和测试脚本中的硬编码路径为 Colab 路径 |
| 数据集管理 | 通过 Google Drive 挂载 + 7z 直接解压分卷，支持断点续传；自动清理 zip 释放空间 |
| 模型获取 | OSTrack 预训练模型和 SDSTrack checkpoint 通过 Drive 快捷方式绕过 Google 下载配额 |
| 长时间运行 | 评测单元格内置 keepalive 线程，防止 Colab 空闲断开；支持断点续跑 |
| 指标计算 | 自动读取 tracking results，计算 **Success AUC** 和 **Precision @ 20px**，直接用于报告 |

### 最终目标
在 VisEvent 测试集上复现 SDSTrack，获得量化的跟踪性能指标，为课程设计报告提供实验数据支撑。

In [ ]:
# ============================================================
# Quick State Check
# Run this anytime to see current progress.
# ============================================================

import os

def check(path):
    return os.path.exists(path)

def dir_items(path):
    return len([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]) if check(path) else 0

MOUNT = "/content/drive"
BASE = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
WS = "/content/sdstrack"

checks = {
    "Google Drive mounted": check(f"{MOUNT}/MyDrive"),
    "SDSTrack cloned": check(f"{WS}/lib"),
    "Patches applied": check(f"{WS}/.patches_applied"),
    "Train extracted": dir_items(f"{BASE}/train") > 0,
    "Test extracted": dir_items(f"{BASE}/test") > 0,
    "Pretrained model": check(f"{WS}/pretrained/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar"),
    "Eval model ready": check(f"{WS}/models/SDSTrack_cvpr2024_rgbe.pth.tar"),
}

print("=" * 50)
print("SDSTrack Setup State")
print("=" * 50)
for name, ok in checks.items():
    print(f"  {'✅' if ok else '⬜'} {name}")

done = sum(checks.values())
print("=" * 50)
print(f"Progress: {done}/{len(checks)}")
if done == len(checks):
    print("All set! Proceed to Phase 4: Evaluation.")
elif not checks["Google Drive mounted"]:
    print("Next: Run Phase 1 → Mount Google Drive")
elif not checks["SDSTrack cloned"]:
    print("Next: Run Phase 1 → Environment Setup")
elif not checks["Train extracted"]:
    print("Next: Run Phase 2 → Extract Train Set")
elif not checks["Test extracted"]:
    print("Next: Run Phase 2 → Extract Test Set")
elif not checks["Eval model ready"]:
    print("Next: Run Phase 3 → Model Preparation")
else:
    print("Next: Continue with remaining setup steps.")

## Phase 1: Environment Setup

Set up the SDSTrack codebase and its dependencies.

**Notes:**
- Colab provides PyTorch 2.x (upstream requires 1.11.0, but we apply compatibility patches).
- All cells are idempotent — safe to re-run.

In [ ]:
# ============================================================
# 1.1 Mount Google Drive
# Idempotent: skips if already mounted.
# ============================================================

import os
from google.colab import drive

MOUNT_POINT = "/content/drive"

if os.path.ismount(MOUNT_POINT) and os.path.exists(f"{MOUNT_POINT}/MyDrive"):
    print("✅ Google Drive already mounted.")
else:
    print("🔌 Mounting Google Drive...")
    drive.mount(MOUNT_POINT, force_remount=False)
    print("✅ Drive mounted successfully!")

In [ ]:
# ============================================================
# 1.2 GPU and CUDA Verification
# Idempotent: pure diagnostics.
# ============================================================

import torch

print("=" * 50)
print("GPU / CUDA Info")
print("=" * 50)

!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

print("\nPyTorch CUDA Info")
print("-" * 50)
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA (PyTorch):   {torch.version.cuda}")
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Quick GPU tensor test
if torch.cuda.is_available():
    x = torch.rand(500, 500).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU tensor test:  OK (shape {y.shape})")
else:
    print("\n⚠️ WARNING: CUDA not available!")

In [ ]:
# ============================================================
# 1.3 Install SDSTrack Dependencies
# Idempotent: checks each package before installing.
# ============================================================

import subprocess
import sys
import importlib

print("Checking dependencies...")

PKGS = [
    ('PyYAML', 'yaml'),
    ('easydict', 'easydict'),
    ('cython', 'cython'),
    ('opencv-python', 'cv2'),
    ('pandas', 'pandas'),
    ('pycocotools', 'pycocotools'),
    ('jpeg4py', 'jpeg4py'),
    ('scipy', 'scipy'),
    ('timm==0.5.4', 'timm'),
    ('tb-nightly', 'tensorboard'),
    ('lmdb', 'lmdb'),
    ('visdom', 'visdom'),
    ('wandb', 'wandb'),
    ('vot-toolkit==0.5.3', 'vot_toolkit'),
    ('vot-trax==3.0.3', 'vot_trax'),
    ('tqdm', 'tqdm'),
]

missing = []
for pkg, mod in PKGS:
    try:
        importlib.import_module(mod.replace('-', '_'))
        print(f"  ✅ {pkg.split('==')[0]}")
    except ImportError:
        print(f"  ⬜ {pkg.split('==')[0]}")
        missing.append(pkg)

if missing:
    print(f"\nInstalling {len(missing)} missing packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("✅ Installation complete.")
else:
    print("\n✅ All dependencies already installed.")

In [ ]:
# ============================================================
# 1.4 Verify Installation
# Idempotent: pure diagnostics.
# ============================================================

import torch
import cv2
import yaml
import timm
import scipy

print("=" * 50)
print("Dependency Versions")
print("=" * 50)
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()} ({torch.version.cuda if torch.cuda.is_available() else 'N/A'})")
print(f"OpenCV:   {cv2.__version__}")
print(f"timm:     {timm.__version__}")
print(f"scipy:    {scipy.__version__}")

if torch.cuda.is_available():
    x = torch.rand(1000, 1000).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU test: OK (result {y.shape})")
else:
    print("\n⚠️ No GPU detected!")

In [ ]:
# ============================================================
# 1.5 Clone SDSTrack & Apply Compatibility Patches
# Idempotent: skips clone/patch if already done.
# ============================================================

import os

WS = "/content/sdstrack"
os.makedirs(WS, exist_ok=True)
os.chdir(WS)

UPSTREAM = os.path.join(WS, ".upstream_cloned")
PATCHED = os.path.join(WS, ".patches_applied")
PATHS = os.path.join(WS, ".paths_fixed")

# --- Clone ---
if os.path.exists(UPSTREAM) and os.path.exists(os.path.join(WS, "lib")):
    print("✅ SDSTrack already cloned.")
else:
    print("🔽 Cloning SDSTrack...")
    !git clone https://github.com/hoqolo/SDSTrack.git .
    with open(UPSTREAM, "w") as f:
        f.write("done")
    print("✅ Clone complete.")

# --- Apply PyTorch 2.x / Python 3.10+ patches ---
if os.path.exists(PATCHED):
    print("✅ Patches already applied.")
else:
    loader = os.path.join(WS, "lib", "train", "data", "loader.py")
    if not os.path.exists(loader):
        print("❌ loader.py not found. Clone may have failed.")
    else:
        print("🔧 Applying compatibility patches...")
        with open(loader, "r") as f:
            c = f.read()

        if "import collections.abc" not in c:
            c = c.replace("import collections", "import collections\nimport collections.abc")

        c = c.replace("from torch._six import string_classes",
                      "try:\n    from torch._six import string_classes\nexcept ImportError:\n    string_classes = (str, bytes)")
        c = c.replace("collections.Mapping", "collections.abc.Mapping")
        c = c.replace("collections.Sequence", "collections.abc.Sequence")

        with open(loader, "w") as f:
            f.write(c)

        with open(PATCHED, "w") as f:
            f.write("done")
        print("✅ Patches applied to loader.py")

# --- Fix paths ---
if os.path.exists(PATHS):
    print("✅ Paths already fixed.")
else:
    print("⚙️ Configuring paths...")
    !python tracking/create_default_local_file.py --workspace_dir . --data_dir ./data --save_dir ./output

    local_py = os.path.join(WS, "lib", "train", "admin", "local.py")
    if os.path.exists(local_py):
        with open(local_py, "r") as f:
            c = f.read()
        c = c.replace(
            "self.visevent_dir = '/home/houxiaojun/Workspace/SDSTrack/data/visevent/train'",
            "self.visevent_dir = '/content/sdstrack/data/visevent/train/train_subset'"
        )
        with open(local_py, "w") as f:
            f.write(c)
        print("✅ Fixed visevent_dir")

    test_script = os.path.join(WS, "RGBE_workspace", "test_rgbe_mgpus.py")
    if os.path.exists(test_script):
        with open(test_script, "r") as f:
            c = f.read()
        c = c.replace(
            "seq_home = '/public/datasets_neo/VisEvent/VisEvent_dataset/testset/test_subset'",
            "seq_home = '/content/sdstrack/data/visevent/test/test_subset'"
        )
        with open(test_script, "w") as f:
            f.write(c)
        print("✅ Fixed test script path")

    with open(PATHS, "w") as f:
        f.write("done")
    print("✅ Paths configured.")

In [ ]:
# ============================================================
# 1.6 Test Imports
# Idempotent: verifies the environment is functional.
# ============================================================

import sys
sys.path.insert(0, "/content/sdstrack")

tests = [
    ("lib.train.data.loader", "LTRLoader"),
    ("lib.test.evaluation", "create_default_local_file_test"),
    ("lib.train.admin.local", "EnvironmentSettings"),
]

all_ok = True
for mod, obj in tests:
    try:
        m = __import__(mod, fromlist=[obj])
        getattr(m, obj)
        print(f"  ✅ {mod}.{obj}")
    except Exception as e:
        print(f"  ❌ {mod}.{obj}: {e}")
        all_ok = False

if all_ok:
    from lib.train.admin import local
    print(f"\n📁 VisEvent train dir: {local.EnvironmentSettings().visevent_dir}")
    print("\n🎉 Environment ready!")
else:
    print("\n⚠️ Some imports failed. Check earlier cells.")

In [ ]:
# ============================================================
# 1.7 Create Dataset Symlinks
# Idempotent: handles existing directories/links safely.
# ============================================================

import os
import shutil

WS = "/content/sdstrack"
DATA_DIR = os.path.join(WS, "data")
os.makedirs(DATA_DIR, exist_ok=True)

DRIVE_DATA = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
link = os.path.join(DATA_DIR, "visevent")

if os.path.islink(link) and os.path.exists(link):
    print(f"✅ Symlink already exists -> {os.readlink(link)}")
elif os.path.isdir(link) and not os.path.islink(link):
    print("⚠️ Directory exists, replacing with symlink...")
    shutil.rmtree(link)
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
        print(f"✅ Symlink created -> {DRIVE_DATA}")
    else:
        print(f"❌ Drive data not found: {DRIVE_DATA}")
elif not os.path.exists(link):
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
        print(f"✅ Symlink created -> {DRIVE_DATA}")
    else:
        print(f"❌ Drive data not found: {DRIVE_DATA}")

# Summary
print("\n📂 Data directory:")
for item in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, item)
    kind = "[LINK]" if os.path.islink(p) else "[DIR]" if os.path.isdir(p) else "[FILE]"
    print(f"  {item:20s} {kind}")

In [ ]:
# ============================================================
# 1.8 Download OSTrack Pretrained Model
# Idempotent: skips if model already exists.
#
# NOTE: The public gdown link often hits quota. Instead:
#  1. Open https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy?usp=sharing
#  2. Right-click 'OSTrack_ep0300.pth.tar' → Add shortcut → My Drive
#  3. Re-run this cell (reads shortcut from mounted Drive)
# ============================================================

import os
import shutil

PRETRAINED_DIR = "/content/sdstrack/pretrained/vitb_256_mae_ce_32x4_ep300"
expected = os.path.join(PRETRAINED_DIR, "OSTrack_ep0300.pth.tar")
shortcut = "/content/drive/MyDrive/OSTrack_ep0300.pth.tar"

if os.path.exists(expected):
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"✅ Model already exists ({mb:.1f} MB)")
elif os.path.exists(shortcut):
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    shutil.copy2(shortcut, expected)
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"✅ Copied from Drive shortcut ({mb:.1f} MB)")
else:
    print("⬜ Model not found.")
    print("\nPlease add a shortcut to your Google Drive:")
    print("  1. Open: https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy?usp=sharing")
    print("  2. Right-click 'OSTrack_ep0300.pth.tar'")
    print("  3. Select 'Organize' → 'Add shortcut' → 'All locations' → 'My Drive' → 'Add'")
    print("  4. Re-run this cell")

## Phase 2: Dataset Preparation

Extract the VisEvent dataset from Google Drive zip archives.

**Prerequisite:** Dataset zip files must be in `MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset/`.

| Set | Files | Size | Est. Time |
|-----|-------|------|-----------|
| Train | `VisEvent_train.zip` + `.z01`–`.z06` | ~130 GB | 2–4 h |
| Test  | `VisEvent_test.zip` + `.z01`–`.z05` | ~102 GB | 1–3 h |

We use `7z` to extract directly from split parts (no merge step needed).

**Important:**
- Extraction writes to Google Drive (not local disk).
- Keep the Colab tab active to prevent idle disconnect.
- All cells are idempotent — will skip if already extracted.

### 2.1 Extract VisEvent Train Set

Extracts to `MyDrive/EvTrack/datasets/VisEvent/train/`.

If the cell seems stuck with no output, check Drive for growing folders — 7z does not print per-file progress.

In [ ]:
# ============================================================
# 2.1 Extract VisEvent Train Set
# Idempotent: skips if train/ already contains sequences.
# ============================================================

import os
import subprocess

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
TRAIN_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/train"

if os.path.exists(TRAIN_DIR) and len([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]) > 0:
    print("✅ Train set already extracted. Skipping.")
    raise SystemExit

if not os.path.exists(VISEVENT_DIR):
    print(f"❌ Zip directory not found: {VISEVENT_DIR}")
    raise SystemExit

zips = [f for f in os.listdir(VISEVENT_DIR) if f.startswith("VisEvent_train")]
if not zips:
    print("❌ No train zip files found.")
    raise SystemExit

# Ensure 7z is available
try:
    subprocess.run(["7z"], capture_output=True, check=True)
except Exception:
    print("📦 Installing p7zip-full...")
    !apt-get update -qq && apt-get install -y -qq p7zip-full

os.makedirs(TRAIN_DIR, exist_ok=True)
os.chdir(VISEVENT_DIR)

print("=" * 50)
print("Extracting VisEvent TRAIN set")
print("=" * 50)
print(f"Source:      {VISEVENT_DIR}")
print(f"Destination: {TRAIN_DIR}")
print("This may take 2–4 hours. Do not interrupt.")
print("=" * 50)

result = subprocess.run(["7z", "x", "VisEvent_train.zip", f"-o{TRAIN_DIR}"], capture_output=False, text=True)

if result.returncode == 0:
    seqs = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    print(f"\n✅ Train extraction complete! ({len(seqs)} sequences)")
else:
    print(f"\n⚠️ 7z exited with code {result.returncode}")

In [ ]:
# ============================================================
# 2.2 Clean Up Train Zip Files
# Idempotent: skips if already deleted.
# ============================================================

import os

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
os.chdir(VISEVENT_DIR)

files = [f for f in os.listdir(".") if f.startswith("VisEvent_train")]
if not files:
    print("✅ No train zip files to delete.")
else:
    total = sum(os.path.getsize(f) / 1e9 for f in files)
    print(f"🗑️ Deleting {len(files)} files ({total:.1f} GB)...")
    for f in files:
        os.remove(f)
        print(f"   ✓ {f}")
    print("\n✅ Cleanup complete.")

### 2.3 Extract VisEvent Test Set

Extracts to `MyDrive/EvTrack/datasets/VisEvent/test/`.

Same procedure as train set.

In [ ]:
# ============================================================
# 2.3 Extract VisEvent Test Set
# Idempotent: skips if test/ already contains sequences.
# ============================================================

import os
import subprocess

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
TEST_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/test"

if os.path.exists(TEST_DIR) and len([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))]) > 0:
    print("✅ Test set already extracted. Skipping.")
    raise SystemExit

zips = [f for f in os.listdir(VISEVENT_DIR) if f.startswith("VisEvent_test")]
if not zips:
    print("❌ No test zip files found.")
    raise SystemExit

os.makedirs(TEST_DIR, exist_ok=True)
os.chdir(VISEVENT_DIR)

print("=" * 50)
print("Extracting VisEvent TEST set")
print("=" * 50)
print(f"Source:      {VISEVENT_DIR}")
print(f"Destination: {TEST_DIR}")
print("This may take 1–3 hours. Do not interrupt.")
print("=" * 50)

result = subprocess.run(["7z", "x", "VisEvent_test.zip", f"-o{TEST_DIR}"], capture_output=False, text=True)

if result.returncode == 0:
    seqs = [d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))]
    print(f"\n✅ Test extraction complete! ({len(seqs)} sequences)")
else:
    print(f"\n⚠️ 7z exited with code {result.returncode}")

In [ ]:
# ============================================================
# 2.4 Clean Up Test Zip Files
# Idempotent: skips if already deleted.
# ============================================================

import os

VISEVENT_DIR = "/content/drive/MyDrive/EvTrack/datasets/VisEvent/VisEvent_dataset"
os.chdir(VISEVENT_DIR)

files = [f for f in os.listdir(".") if f.startswith("VisEvent_test")]
if not files:
    print("✅ No test zip files to delete.")
else:
    total = sum(os.path.getsize(f) / 1e9 for f in files)
    print(f"🗑️ Deleting {len(files)} files ({total:.1f} GB)...")
    for f in files:
        os.remove(f)
        print(f"   ✓ {f}")
    print("\n✅ Cleanup complete.")

In [ ]:
# ============================================================
# 2.5 Verify Extracted Structure
# Idempotent: pure diagnostics.
# ============================================================

import os

BASE = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"

for split in ["train", "test"]:
    split_dir = os.path.join(BASE, split)
    print(f"\n{'='*50}")
    print(f"{split.upper()} SET")
    print("="*50)

    if not os.path.exists(split_dir):
        print("⬜ Not yet extracted")
        continue

    top = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
    if not top:
        print("❌ No directories found")
        continue

    # Check for wrapper folders (train_subset / test_subset)
    first_top = os.path.join(split_dir, top[0])
    sub = [d for d in os.listdir(first_top) if os.path.isdir(os.path.join(first_top, d))]

    if sub:
        sequences = sub
        wrapper = top[0]
        print(f"📁 Wrapper:   {wrapper}")
    else:
        sequences = top
        wrapper = None
        print(f"📁 Direct structure")

    print(f"📁 Sequences: {len(sequences)}")

    if sequences:
        seq_path = os.path.join(split_dir, wrapper, sequences[0]) if wrapper else os.path.join(split_dir, sequences[0])
        print(f"\nExample '{sequences[0]}':")
        for exp in ["vis_imgs", "event_imgs", "groundtruth.txt"]:
            ok = os.path.exists(os.path.join(seq_path, exp))
            print(f"  {exp:20s} {'✅' if ok else '❌'}")

print("\n" + "="*50)
print("Verification complete")
print("="*50)

## Phase 3: Model Preparation

Prepare the pretrained checkpoint for evaluation and apply PyTorch 2.x compatibility patches.

In [ ]:
# ============================================================
# 3.1 Prepare Model Checkpoint
# Idempotent: skips if already set up.
# ============================================================

import os
import shutil

WS = "/content/sdstrack"
os.chdir(WS)

MODEL_DIR = os.path.join(WS, "models")
CKPT_DIR = os.path.join(WS, "output", "checkpoints", "train", "sdstrack", "cvpr2024_rgbe")
SYMLINK = os.path.join(MODEL_DIR, "SDSTrack_cvpr2024_rgbe.pth.tar")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists(SYMLINK):
    mb = os.path.getsize(os.path.realpath(SYMLINK)) / (1024 * 1024)
    print(f"✅ Model ready ({mb:.1f} MB)")
    print(f"   {SYMLINK}")
    raise SystemExit

# Search for shortcuts in Drive
NAMES = ["SDSTrack_cvpr2024_rgbe.pth.tar", "SDSTrack_ep0050.pth.tar"]
found = None
for name in NAMES:
    for path in [f"/content/drive/MyDrive/{name}", f"/content/drive/MyDrive/SDSTrack_models/{name}"]:
        if os.path.exists(path):
            found = path
            break
    if found:
        break

if found:
    dest = os.path.join(CKPT_DIR, "SDSTrack_ep0050.pth.tar")
    shutil.copy2(found, dest)
    os.symlink(dest, SYMLINK)
    mb = os.path.getsize(dest) / (1024 * 1024)
    print(f"✅ Copied from Drive ({mb:.1f} MB)")
    print(f"   {SYMLINK} -> {dest}")
else:
    print("⬜ Model checkpoint not found.")
    print("\nPlease add a shortcut to your Google Drive:")
    print("  1. Find the model file in the shared folder")
    print("  2. Right-click → 'Organize' → 'Add shortcut' → 'My Drive'")
    print("  3. Re-run this cell")

In [ ]:
# ============================================================
# 3.2 Apply PyTorch 2.x Compatibility Patches
# PyTorch 2.6+ defaults to weights_only=True, breaking old checkpoints.
# Idempotent: safe to re-run.
# ============================================================

import os

WS = "/content/sdstrack"
PATCHES = [
    ("lib/test/tracker/sdstrack.py",
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu")',
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu", weights_only=False)'),
    ("lib/train/trainers/base_trainer.py",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"),
]

for filepath, old, new in PATCHES:
    full = os.path.join(WS, filepath)
    if not os.path.exists(full):
        print(f"❌ File not found: {filepath}")
        continue

    with open(full, "r") as f:
        content = f.read()

    if old in content:
        content = content.replace(old, new)
        with open(full, "w") as f:
            f.write(content)
        print(f"✅ Patched {filepath}")
    else:
        print(f"⬜ Already patched or not found: {filepath}")

print("\nDone. Proceed to Phase 4: Evaluation.")

## Phase 4: Evaluation

Run SDSTrack on the VisEvent test set and compute metrics.

**Expected time:** 30–90 minutes (single-threaded for stability).

**Output:**
- Tracking results: `RGBE_workspace/results/VisEvent/cvpr2024_rgbe/`
- Metrics: Success AUC, Precision @ 20px

In [ ]:
# ============================================================
# 4.1 Create testlist.txt and trainlist.txt
# Idempotent: skips if files already exist.
# ============================================================

import os

TEST_DIR = "/content/sdstrack/data/visevent/test/test_subset"
TRAIN_DIR = "/content/sdstrack/data/visevent/train/train_subset"

for dirname, listname in [(TEST_DIR, "testlist.txt"), (TRAIN_DIR, "trainlist.txt")]:
    path = os.path.join(dirname, listname)
    if os.path.exists(path):
        with open(path, "r") as f:
            n = len(f.readlines())
        print(f"✅ {listname} exists ({n} sequences)")
    else:
        seqs = sorted([d for d in os.listdir(dirname) if os.path.isdir(os.path.join(dirname, d))])
        with open(path, "w") as f:
            for s in seqs:
                f.write(s + "\n")
        print(f"✅ Created {listname} ({len(seqs)} sequences)")

In [ ]:
# ============================================================
# 4.2 Run Evaluation on VisEvent Test Set
# - Streams output in real-time
# - Keepalive thread prevents idle disconnect
# - threads=1 for stability
# - Resumable: skips completed sequences
# ============================================================

import os
import subprocess
import threading
import time

WS = "/content/sdstrack"
os.chdir(WS)

MODEL_PATH = "./models/SDSTrack_cvpr2024_rgbe.pth.tar"
if not os.path.exists(MODEL_PATH):
    print("❌ Model not found. Run Phase 3 first.")
    raise SystemExit

RESULTS_DIR = "./RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
existing = 0
if os.path.exists(RESULTS_DIR):
    existing = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])
    if existing > 0:
        print(f"⏮️ Found {existing} existing results (will skip if resuming)")

print("=" * 50)
print("Running SDSTrack Evaluation")
print("=" * 50)
print(f"Model:    {MODEL_PATH}")
print(f"Test set: ./data/visevent/test/test_subset")
print(f"Expected: 30–90 min (77 sequences, threads=1)")
print("=" * 50)

# Keepalive thread
keepalive_running = True
def keepalive():
    start = time.time()
    while keepalive_running:
        time.sleep(30)
        elapsed = time.time() - start
        current = 0
        if os.path.exists(RESULTS_DIR):
            current = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])
        print(f"\n⏱️  [{elapsed/60:.1f} min] Completed: {current}/77 (keepalive)\n", flush=True)

t = threading.Thread(target=keepalive, daemon=True)
t.start()

# Run evaluation
process = subprocess.Popen(
    ["python", "./RGBE_workspace/test_rgbe_mgpus.py",
     "--script_name", "sdstrack",
     "--num_gpus", "1",
     "--threads", "1",
     "--epoch", "50",
     "--yaml_name", "cvpr2024_rgbe"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted by user")

process.wait()
keepalive_running = False

final = 0
if os.path.exists(RESULTS_DIR):
    final = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])

print("\n" + "=" * 50)
if process.returncode == 0 and final >= 77:
    print(f"✅ Evaluation complete! ({final}/77 sequences)")
elif final > 0:
    print(f"⚠️ Partial results: {final}/77 sequences")
    print("Re-run this cell to resume (completed sequences are skipped).")
else:
    print(f"❌ Evaluation failed (exit code: {process.returncode})")
print(f"Results: {RESULTS_DIR}")
print("=" * 50)

In [ ]:
# ============================================================
# 4.3 Compute Evaluation Metrics
# Computes Success (AUC) and Precision (20px) from tracking results.
# ============================================================

import os
import numpy as np
import glob

RESULTS_DIR = "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
GT_BASE = "/content/sdstrack/data/visevent/test/test_subset"

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

def compute_metrics(results_dir, gt_base):
    if not os.path.exists(results_dir):
        print("❌ Results directory not found. Run Phase 4.2 first.")
        return

    files = sorted(glob.glob(os.path.join(results_dir, "*.txt")))
    if not files:
        print("❌ No result files found.")
        return

    print(f"Processing {len(files)} result files...")

    all_ious, all_dists = [], []

    for res_file in files:
        seq = os.path.basename(res_file).replace(".txt", "")
        gt_file = os.path.join(gt_base, seq, "groundtruth.txt")

        if not os.path.exists(gt_file):
            print(f"⚠️  Skipping {seq} (no groundtruth)")
            continue

        try:
            pred = np.loadtxt(res_file, delimiter=",")
            gt = np.loadtxt(gt_file, delimiter=",")
        except Exception:
            print(f"⚠️  Skipping {seq} (load error)")
            continue

        if pred.ndim == 1:
            pred = pred.reshape(1, -1)
        if gt.ndim == 1:
            gt = gt.reshape(1, -1)

        n = min(len(pred), len(gt))
        pred, gt = pred[:n], gt[:n]

        for p, g in zip(pred, gt):
            all_ious.append(compute_iou(p, g))
            pcx, pcy = p[0] + p[2]/2, p[1] + p[3]/2
            gcx, gcy = g[0] + g[2]/2, g[1] + g[3]/2
            all_dists.append(np.sqrt((pcx - gcx)**2 + (pcy - gcy)**2))

    if not all_ious:
        print("No valid data to compute metrics.")
        return

    all_ious = np.array(all_ious)
    all_dists = np.array(all_dists)

    # Success AUC
    thresholds = np.arange(0, 1.05, 0.05)
    success_rates = [np.mean(all_ious >= t) for t in thresholds]
    auc = np.mean(success_rates)

    # Precision @ 20px
    prec_20 = np.mean(all_dists <= 20)

    print("\n" + "=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Success AUC:      {auc:.4f}")
    print(f"Precision (20px): {prec_20:.4f}")
    print(f"Total frames:     {len(all_ious)}")
    print("=" * 50)

    return {"auc": auc, "prec_20": prec_20}

metrics = compute_metrics(RESULTS_DIR, GT_BASE)

## Appendix: Troubleshooting & Notes

### Common Issues

**1. `torch.load` weights_only error**
- Already fixed in Phase 3.2. If you see this error, re-run Phase 3.2.

**2. Missing `testlist.txt`**
- Already handled in Phase 4.1. The cell auto-creates it if missing.

**3. Evaluation crashes with multiprocessing**
- Phase 4.2 uses `threads=1` for stability. It is slower but reliable.

**4. Colab disconnects during long extraction**
- Keep the browser tab active.
- The evaluation cell has a built-in keepalive thread.

**5. Model not found**
- The OSTrack pretrained model and SDSTrack checkpoint must be added as **shortcuts** to your Google Drive (see Phase 1.8 and Phase 3.1).

### File Locations

| Item | Path |
|------|------|
| Workspace | `/content/sdstrack` |
| Data symlink | `/content/sdstrack/data/visevent` |
| Pretrained model | `/content/sdstrack/pretrained/...` |
| Checkpoint | `/content/sdstrack/output/checkpoints/...` |
| Results | `/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe/` |